# 🎬 HypKraft AI Studio — Kaggle Remote GPU Worker
> **Free Tier:** 2× Nvidia T4 (16 GB VRAM each) · 30 GPU-hrs/week · Resets every Saturday

## ⚠️ GPU Hours — How They Work
- ⏱ **Hours count** from the moment this session starts until it's stopped
- ✅ **Auto-shutdown**: This notebook stops itself after **30 min of idle** (no generation requests) to save your quota
- 🔁 **On-demand**: Start it only when you need GPU generation, stop when done

## Quick Start
1. Enable **Accelerator → GPU T4 x2** in Session Options
2. Enable **Internet → On** in Session Options  
3. **Run All** (▶▶) — takes ~4 min to install packages
4. Copy the `*.trycloudflare.com` URL from Cell 5 output
5. Paste URL into HypKraft → MCP → Kaggle Tunnel field
6. **Stop the session when done** to save quota

In [ ]:
# ── Cell 1: System Check ──────────────────────────────────────────────────────
import subprocess, torch, os, time
SESSION_START = time.time()

print('🖥  GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '⚠ No GPU detected!')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'💾 VRAM: {round(props.total_memory / 1e9, 1)} GB')
print('🐍 PyTorch:', torch.__version__)
print(f'⏱  Session started at: {time.strftime("%H:%M UTC", time.gmtime())}')
print('⏹  Auto-shutdown: 30 min idle timeout active')

In [ ]:
# ── Cell 2: Install Dependencies ──────────────────────────────────────────────
print('📦 Installing packages (~3 min on first run)...')
import subprocess

# Cloudflared
subprocess.run(['wget', '-q', '-O', '/usr/local/bin/cloudflared',
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'],
    check=True)
subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)
print('✅ cloudflared ready')

# Server + audio
subprocess.run(['pip', 'install', '-q', 'fastapi', 'uvicorn', 'python-multipart',
    'aiofiles', 'pydantic', 'requests', 'audiocraft'], check=True)
print('✅ AudioCraft + FastAPI ready')

# Image generation
subprocess.run(['pip', 'install', '-q', 'diffusers', 'accelerate',
    'transformers', 'safetensors', 'xformers'], check=True)
print('✅ Diffusers + SDXL ready')

print('\n🎉 All packages installed!')

In [ ]:
# ── Cell 3: Write HypKraft GPU Server (with idle auto-shutdown) ───────────────
server_code = r'''
import os, io, base64, threading, time, gc
from pathlib import Path
from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel
import torch

OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

app = FastAPI(title="HypKraft Kaggle GPU Worker", version="2.0.0")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_models = {}
_lock = threading.Lock()

# ── Idle Shutdown Tracker ─────────────────────────────────────────────────────
IDLE_TIMEOUT_SECONDS = 30 * 60  # 30 minutes
_last_request_time = time.time()
_session_start = time.time()

def update_activity():
    global _last_request_time
    _last_request_time = time.time()

def idle_watchdog():
    """Stops the session after IDLE_TIMEOUT_SECONDS of no generation requests."""
    while True:
        time.sleep(60)
        idle_secs = time.time() - _last_request_time
        session_mins = round((time.time() - _session_start) / 60, 1)
        print(f"[Watchdog] Session: {session_mins}min | Idle: {round(idle_secs/60, 1)}min / {IDLE_TIMEOUT_SECONDS//60}min limit")
        if idle_secs > IDLE_TIMEOUT_SECONDS:
            print(f"[Watchdog] ⏹ Idle timeout reached — shutting down to save GPU quota!")
            os._exit(0)  # Hard stop — Kaggle session ends

threading.Thread(target=idle_watchdog, daemon=True).start()

# ── Lazy Model Loaders ────────────────────────────────────────────────────────
def load_musicgen(size="large"):
    if "musicgen" not in _models:
        from audiocraft.models import MusicGen
        print(f"Loading MusicGen-{size}...")
        _models["musicgen"] = MusicGen.get_pretrained(size)
        _models["musicgen"].set_generation_params(duration=15)
    return _models["musicgen"]

def load_audiogen():
    if "audiogen" not in _models:
        from audiocraft.models import AudioGen
        print("Loading AudioGen-medium...")
        _models["audiogen"] = AudioGen.get_pretrained("facebook/audiogen-medium")
        _models["audiogen"].set_generation_params(duration=8)
    return _models["audiogen"]

def load_sdxl():
    if "sdxl" not in _models:
        from diffusers import StableDiffusionXLPipeline
        print("Loading SDXL-Turbo...")
        pipe = StableDiffusionXLPipeline.from_pretrained(
            "stabilityai/sdxl-turbo",
            torch_dtype=torch.float16, variant="fp16"
        ).to(DEVICE)
        pipe.enable_xformers_memory_efficient_attention()
        _models["sdxl"] = pipe
    return _models["sdxl"]

# ── Request Models ────────────────────────────────────────────────────────────
class AudioRequest(BaseModel):
    prompt: str
    duration: float = 10.0
    type: str = "music"
    category: str = "cinematic"

class ImageRequest(BaseModel):
    prompt: str
    width: int = 1024
    height: int = 576
    steps: int = 4
    guidance_scale: float = 0.0

# ── Endpoints ─────────────────────────────────────────────────────────────────
@app.get("/")
def root():
    return {"service": "HypKraft Kaggle GPU Worker", "version": "2.0", "status": "online"}

@app.get("/health")
def health():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    vram_free = round((torch.cuda.mem_get_info()[0] if torch.cuda.is_available() else 0) / 1e9, 2)
    vram_total = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else 0
    idle_mins = round((time.time() - _last_request_time) / 60, 1)
    session_mins = round((time.time() - _session_start) / 60, 1)
    return {
        "status": "online",
        "gpu": gpu_name,
        "vram_free_gb": vram_free,
        "vram_total_gb": vram_total,
        "loaded_models": list(_models.keys()),
        "session_minutes": session_mins,
        "idle_minutes": idle_mins,
        "auto_shutdown_at_idle_minutes": IDLE_TIMEOUT_SECONDS // 60,
        "device": DEVICE
    }

@app.get("/session/stats")
def session_stats():
    session_hrs = round((time.time() - _session_start) / 3600, 2)
    idle_mins = round((time.time() - _last_request_time) / 60, 1)
    return {
        "session_hours_used": session_hrs,
        "idle_minutes": idle_mins,
        "auto_shutdown_in_minutes": max(0, round((IDLE_TIMEOUT_SECONDS - (time.time() - _last_request_time)) / 60, 1)),
        "models_loaded": list(_models.keys())
    }

@app.post("/session/keepalive")
def keepalive():
    """Reset the idle timer without doing any generation (call this to keep session alive)."""
    update_activity()
    return {"ok": True, "idle_reset": True}

@app.post("/generate/audio")
async def generate_audio(req: AudioRequest):
    update_activity()
    try:
        with _lock:
            if req.type in ("sfx", "ambient"):
                model = load_audiogen()
                model.set_generation_params(duration=min(req.duration, 8.0))
                model_name = "AudioCraft-AudioGen-Medium"
            else:
                model = load_musicgen()
                model.set_generation_params(duration=min(req.duration, 30.0))
                model_name = "AudioCraft-MusicGen-Large"

            wav = model.generate([req.prompt])
            from audiocraft.data.audio import audio_write
            out = OUTPUT_DIR / f"audio_{int(time.time())}"
            audio_write(str(out), wav[0].cpu(), model.sample_rate,
                        strategy="loudness", loudness_compressor=True)
            wav_path = str(out) + ".wav"

        with open(wav_path, "rb") as f:
            audio_b64 = base64.b64encode(f.read()).decode()

        return {
            "status": "completed", "prompt": req.prompt, "model": model_name,
            "duration_s": req.duration, "audio_base64": audio_b64, "format": "wav"
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/generate/image")
async def generate_image(req: ImageRequest):
    update_activity()
    try:
        with _lock:
            pipe = load_sdxl()
            result = pipe(prompt=req.prompt, width=req.width, height=req.height,
                          num_inference_steps=req.steps, guidance_scale=req.guidance_scale)
            img = result.images[0]
        buffer = io.BytesIO()
        img.save(buffer, format="PNG")
        img_b64 = base64.b64encode(buffer.getvalue()).decode()
        return {
            "status": "completed", "prompt": req.prompt, "model": "SDXL-Turbo",
            "image_base64": img_b64, "format": "png"
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.delete("/session/stop")
def stop_session():
    """Gracefully stop this GPU session to save quota."""
    threading.Thread(target=lambda: (time.sleep(2), os._exit(0)), daemon=True).start()
    return {"ok": True, "message": "Session stopping in 2 seconds — quota saved!"}
'''

with open('hypkraft_server.py', 'w') as f:
    f.write(server_code)
print('✅ Server code written (with 30-min idle auto-shutdown)')

In [ ]:
# ── Cell 4: Launch FastAPI Server ─────────────────────────────────────────────
import subprocess, time, requests

server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'hypkraft_server:app', '--host', '0.0.0.0', '--port', '8765'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

print('⏳ Starting FastAPI server...')
for i in range(20):
    time.sleep(1)
    try:
        r = requests.get('http://127.0.0.1:8765/health', timeout=2)
        if r.status_code == 200:
            d = r.json()
            print(f'✅ Server online!')
            print(f'   GPU: {d["gpu"]} | VRAM: {d["vram_total_gb"]} GB')
            print(f'   Auto-shutdown: {d["auto_shutdown_at_idle_minutes"]} min idle timeout')
            break
    except: pass
else:
    print('⚠ Server slow to start — check logs')

In [ ]:
# ── Cell 5: Open Cloudflare Tunnel ── COPY URL TO HYPKRAFT ────────────────────
# ⏱ GPU hours STOP counting when you stop this session.
# 🛑 To stop: click the ⏹ Stop button or call DELETE /session/stop via HypKraft.
import subprocess, threading, time

tunnel_url = None

def run_tunnel():
    global tunnel_url
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8765'],
        stderr=subprocess.PIPE, stdout=subprocess.PIPE
    )
    for line in proc.stderr:
        decoded = line.decode('utf-8', errors='ignore').strip()
        if 'trycloudflare.com' in decoded:
            for part in decoded.split():
                if 'trycloudflare.com' in part:
                    tunnel_url = part.strip('|').strip()
                    print(f'\n' + '='*62)
                    print(f'🌐 KAGGLE GPU TUNNEL URL (copy to HypKraft):')
                    print(f'   {tunnel_url}')
                    print('='*62)
                    print('⏱  GPU hours counting — stop session when done!')
                    break

threading.Thread(target=run_tunnel, daemon=True).start()
print('⏳ Starting Cloudflare tunnel...')

# Status loop — prints GPU quota every 10 min, auto-exits if server dies
import requests
tick = 0
try:
    while True:
        time.sleep(30)
        tick += 1
        try:
            r = requests.get('http://127.0.0.1:8765/session/stats', timeout=5)
            s = r.json()
            if tick % 20 == 0:  # Every 10 min
                print(f'⏱  Session: {s["session_hours_used"]}h used | '
                      f'Idle: {s["idle_minutes"]}min | '
                      f'Auto-stop in: {s["auto_shutdown_in_minutes"]}min')
        except:
            print('⏹ Server stopped — session ending')
            break
except KeyboardInterrupt:
    print('⏹ Stopped by user — GPU quota saved!')